# From a Single Linear Layer to a CNN on CIFAR-10

This short notebook shows the essential deep-learning workflow:

1. load data;
2. define a model;
3. choose a loss and optimizer;
4. train with forward pass, backpropagation, and parameter updates;
5. evaluate on unseen images.

We use fixed settings and compare only two models. There is no hyperparameter search in this tutorial.

In [1]:
import time

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

torch.manual_seed(598)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## 1. Load CIFAR-10

CIFAR-10 contains 32 × 32 RGB images from 10 classes. `ToTensor` converts each image to a tensor with shape `[3, 32, 32]` and values in `[0, 1]`. A smaller fixed subset keeps this demonstration quick on a CPU.

In [ ]:
transform = transforms.ToTensor()

full_train_set = datasets.CIFAR10("./data", train=True, download=True, transform=transform)
full_test_set = datasets.CIFAR10("./data", train=False, download=True, transform=transform)

# Fixed subsets are used only to make the tutorial run quickly.
train_set = Subset(full_train_set, range(10_000))
test_set = Subset(full_test_set, range(2_000))

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False)

images, labels = next(iter(train_loader))
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

## 2. Define two models

The linear classifier flattens every image and connects every input value directly to 10 output logits. It does not explicitly use spatial structure.

The CNN learns local filters and reuses each filter across the image. Pooling reduces spatial resolution while later layers combine increasingly complex features. Both models return **logits**, not probabilities.

In [ ]:
class SingleLayerClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(3 * 32 * 32, 10)

    def forward(self, images):
        flattened = images.flatten(start_dim=1)
        return self.linear(flattened)


class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Linear(64 * 8 * 8, 10)

    def forward(self, images):
        features = self.features(images)
        return self.classifier(features.flatten(start_dim=1))


linear_model = SingleLayerClassifier()
cnn_model = SmallCNN()
print(linear_model)
print(cnn_model)

## 3. Build the training and evaluation loops

For each training batch we clear old gradients, compute predictions and loss, backpropagate, and update the parameters. The cross-entropy loss is implemented directly from the logits using the numerically stable log-sum-exp form. Evaluation disables gradient tracking because no parameter update is needed.

In [ ]:
def cross_entropy_loss(logits, labels):
    # For each example: log(sum(exp(logits))) - logit of the correct class.
    # torch.logsumexp computes the first term without numerical overflow.
    log_normalizer = torch.logsumexp(logits, dim=1)
    correct_class_logits = logits.gather(1, labels.unsqueeze(1)).squeeze(1)
    return (log_normalizer - correct_class_logits).mean()


def train_one_epoch(model, data_loader, loss_function, optimizer):
    model.train()
    total_loss = 0.0
    total_examples = 0

    for images, labels in data_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = loss_function(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total_examples += labels.size(0)

    return total_loss / total_examples


def evaluate(model, data_loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)
            predictions = model(images).argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return correct / total

## 4. Train both models with the same fixed setup

Cross-entropy is appropriate for mutually exclusive classes. Adam performs the parameter updates here. These values are fixed for a short demonstration; they are not a parameter-selection exercise.

In [ ]:
def train_model(name, model):
    model = model.to(device)
    loss_function = cross_entropy_loss
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    start = time.perf_counter()
    history = {"epoch": [], "loss": [], "accuracy": []}

    for epoch in range(1, 4):
        train_loss = train_one_epoch(model, train_loader, loss_function, optimizer)
        test_accuracy = evaluate(model, test_loader)
        history["epoch"].append(epoch)
        history["loss"].append(train_loss)
        history["accuracy"].append(test_accuracy)
        print(f"{name:12s} | epoch {epoch} | loss={train_loss:.4f} | accuracy={test_accuracy:.2%}")

    elapsed = time.perf_counter() - start
    return {
        "model": name,
        "accuracy": history["accuracy"][-1],
        "seconds": elapsed,
        "history": history,
    }


results = [
    train_model("Single layer", linear_model),
    train_model("Small CNN", cnn_model),
]

## 5. Compare the results

Accuracy measures predictive performance, while elapsed time shows part of the computational cost. Your exact numbers can vary slightly across machines.

In [ ]:
for result in results:
    print(
        f"{result['model']:12s} | "
        f"accuracy={result['accuracy']:.2%} | "
        f"training time={result['seconds']:.1f} seconds"
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for result in results:
    history = result["history"]
    axes[0].plot(history["epoch"], history["loss"], marker="o", label=result["model"])
    axes[1].plot(history["epoch"], history["accuracy"], marker="o", label=result["model"])

axes[0].set(title="Training Loss by Epoch", xlabel="Epoch", ylabel="Cross-Entropy Loss")
axes[1].set(title="Test Accuracy by Epoch", xlabel="Epoch", ylabel="Accuracy")
axes[1].set_ylim(0, 1)

for axis in axes:
    axis.set_xticks(results[0]["history"]["epoch"])
    axis.grid(alpha=0.3)
    axis.legend()

plt.tight_layout()
plt.show()

## What to notice

- Both models use the same data, loss function, optimizer, and training loop.
- The main change is the model architecture.
- The linear model treats pixels as one long vector.
- The CNN preserves spatial structure and learns local features.
- A more expressive model can improve accuracy, but usually needs more computation.

This notebook is an introduction and is not a replacement for the required implementations and experiments in the assignment.